In [3]:
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple

from matplotlib.pyplot import plt
import seaborn as sns

ModuleNotFoundError: No module named 'matplotlib'

In [2]:
gfdist_mean = 0.5
gfdist_std = 0.0514
goals_per_game_leagueavg = 5.85
home_ice_advantage = 0.22

num_simulated_seasons = 17
n_bootstrap_samples = 10000
bootstrap_sample_size = 40

In [ ]:
full_season_schedule = pd.read_csv('nhl_schedule_2025.csv')

In [ ]:
# This is the module that randomly draws from the talent distribution for all 32 teams

def assign_team_talent(
    gfdist_mean: float,
    gfdist_std: float,
    goals_per_game_leagueavg: float
    ) -> pd.DataFrame:

    """
    Assigns a random GF% representing true team talent to each NHL team, and calculates average GF and GA per game.

    Arguments:
        gfdist_mean: Mean of the normal distribution for GF%
        gfdist_std: Standard deviation of the normal distribution for GF%
        goals_per_game_leagueavg: Average number of goals per game (GF + GA) in the league

    Returns:
        A Dataframe with Team, GF%, average goals for per game, and average goals against per game
    """

    # Official 3-letter codes for all 32 NHL teams in 2024-25
    teams = [
        'ANA', 'BOS', 'BUF', 'CGY', 'CAR', 'CHI', 'COL', 'CBJ', 'DAL', 'DET', 'EDM', 'FLA', 'LAK', 'MIN', 'MTL', 'NSH',
        'NJD', 'NYI', 'NYR', 'OTT', 'PHI', 'PIT', 'SEA', 'SJS', 'STL', 'TBL', 'TOR', 'UTA', 'VAN', 'VEG', 'WPG', 'WSH'
        ]

    # Assign a randomly chosen GF% to each team, with values restricted to fall between 0 and 1
    gfpct_raw = np.random.normal(loc=gfdist_mean, scale=gfdist_std, size=len(teams))
    gfpct = np.clip(gfpct_raw, 0.0, 1.0)

    # Ranking for GF%
    gfpct_rank = gfpct.rank(method='min', ascending=False).astype(int)

    # Calculate average goals for and goals against per game for each team, using the league average
    gf_game = goals_per_game_leagueavg * gfpct
    ga_game = goals_per_game_leagueavg * (1-gfpct)

    # Create Dataframe to hold this information (CAN THIS BE ADDED TO BY OTHER MODULES????)
    team_data = pd.DataFrame({
        'Team': teams,
        'GF_percent': gfpct,
        'GF_percent_rank': gfpct_rank,
        'GF_per_game': gf_game,
        'GA_per_game': ga_game
    })

    # Set 'Team' as the index for lookup in other modules
    return team_data.set_index('Team')
    

In [ ]:
# Module for simulating a regular season game

def simulate_regular_season_game(
    home_team: 'str',
    away_team: 'str',
    team_averages: pd.DataFrame,
    home_ice_adv: float
) -> Dict:

    """
    Simulates a single regular season game using a Poisson process. Each team's expected GF is adjusted for the opponent's GA.

    Arguments:
        home_team: 3-letter code representing the home team
        away_team: 3-letter code representing the away team
        team_averages: DataFrame with team talent and scoring averages from assign_team_talent
        home_ice_adv: Goal value added to the home team's expected GF

    Returns:
        A dictionary with the game's outcome details.
    """

    # Get baseline averages for each team's scoring
    home_gf = team_averages.loc[home_team, 'GF_per_game']
    home_ga = team_averages.loc[home_team, 'GA_per_game']
    away_gf = team_averages.loc[away_team, 'GF_per_game']
    away_ga = team_averages.loc[away_team, 'GA_per_game']

    # Calculated expected goals (lambda) for each team
    home_gf_lambda = ((home_gf + away_ga) / 2) + home_ice_adv
    away_gf_lambda = (away_gf + home_ga) / 2

    # Simulate score as a Poisson process
    home_score = np.random.poisson(lam=home_gf_lambda)
    away_score = np.random.poisson(lam=away_gf_lambda)

    # Determine the winner and allocate points
    home_pts, away_pts = 0, 0

    if home_score > away_score:
        home_pts, away_pts = 2, 0
    elif away_score > home_score:
        home_pts, away_pts = 0, 2
    else: # Tie after regulation, assign a winner randomly
        if np.random.rand() < 0.5:
            home_pts, away_pts = 2, 1
        else:
            home_pts, away_pts = 1, 2

    return {
        'Home': home_team,
        'Away': away_team,
        'Home_GF': home_score,
        'Away_GF': away_score,
        'Home_Pts': home_pts,
        'Away_Pts': away_pts
    }
    

In [ ]:
# Module for simulating a regular season. The schedule from 2024-25 is used.

def simulate_regular_season(
    full_season_schedule: pd.DataFrame,
    team_averages: pd.DataFrame,
    home_ice_adv: float
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    """
    Simulates a full NHL regular season using the 2024-25 schedule.

    Arguments:
        team_averages: DataFrame from module 1.
        home_ice_adv: Goal value added to home team's expected GF.

    Returns:
        A tuple of 2 DataFrames:
        1. Standings: Team, GP, Points, GF, GA
        2. Game log: Record of each game played.
    """

    # Get the list of all teams from team_averages
    teams = team_averages.index.tolist()

    # Initialize a DataFrame for standings
    standings = pd.DataFrame(
        0,
        index = teams,
        columns = ['GP', 'PTS', 'W', 'L', 'OTL', 'GF', 'GA', 'GD']
    )

    game_log = []

    # Simulate all the games by iterating over the schedule DataFrame
    for index, row in full_season_schedule.iterrows():
        home_team = row['HOME']
        away_team = row['VISITOR']
        
        # Skip if team data is missing for some reason
        if home_team not in teams or away_team not in teams:
            continue

        result = simulate_regular_season_game(home_team, away_team, team_averages, home_ice_adv)

        game_log.append(result)

        # Update the standings
        standings.loc[home_team, 'GP'] += 1
        standings.loc[away_team, 'GP'] += 1

        standings.loc[home_team, 'GF'] += result['Home_GF']
        standings.loc[away_team, 'GF'] += result['Away_GF']
        standings.loc[home_team, 'GA'] += result['Away_GF']
        standings.loc[away_team, 'GA'] += result['Home_GF']
        standings.loc[home_team, 'GD'] += (result['Home_GF'] - result['Away_GF'])
        standings.loc[away_team, 'GD'] += (result['Away_GF'] - result['Home_GF'])

        standings.loc[home_team, 'PTS'] += result['Home_Pts']
        standings.loc[away_team, 'PTS'] += result['Away_Pts']

        # Classify wins and losses based on point distribution
        if result['Home_Pts'] == 2 and result['Away_Pts'] == 0:
            standings.loc[home_team, 'W'] += 1
            standings.loc[away_team, 'L'] += 1
        elif result['Home_Pts'] == 0 and result['Away_Pts'] == 2:
            standings.loc[home_team, 'L'] += 1
            standings.loc[away_team, 'W'] += 1
        elif result['Home_Pts'] == 2 and result['Away_Pts'] == 1:
            standings.loc[home_team, 'W'] += 1
            standings.loc[away_team, 'OTL'] += 1
        elif result['Home_Pts'] == 1 and result['Away_Pts'] == 2:
            standings.loc[home_team, 'OTL'] += 1
            standings.loc[away_team, 'W'] += 1

    return standings.sort_values(by='PTS', ascending=False), pd.DataFrame(game_log) # Consider losing the game log when doing the full simulation
    

In [ ]:
def run_bootstrap_simulation(simulated_game_logs: pd.DataFrame, n_samples: int, sample_size: int):
    """
    Performs a bootstrapping simulation on simulated NHL game data to calculate GF%.

    Args:
        simulated_game_logs (pd.DataFrame): A DataFrame containing all simulated game logs
                                            from multiple seasons, with 'SEASON', 'Home_Team', 'Away_Team',
                                            'Home_GF', and 'Away_GF' columns.
        n_samples (int): The number of bootstrap samples to generate for each team.
        sample_size (int): The number of games to randomly sample for each bootstrap.

    Returns:
        pandas.DataFrame: A DataFrame containing the calculated GF% for each bootstrap
                          sample, along with the corresponding team and season.
    """
    # Initialize a list to hold all the results before concatenating into a single DataFrame.
    # This is more memory-efficient than appending to a DataFrame in a loop.
    all_bootstrap_samples = []

    print(f"Starting bootstrapping on simulated data ({n_samples} samples per team, {sample_size} games per sample ...\n")

    # Prepare unified game log for easier processing
    home_games = simulated_game_logs[['SEASON', 'Home_Team', 'Home_GF', 'Away_GF']].copy()
    home_games.columns = ['SEASON', 'TEAM', 'GF', 'GA']

    away_games = simulated_game_logs[['SEASON', 'Away_Team', 'Away_GF', 'Home_GF']].copy()
    away_games.columns = ['SEASON', 'TEAM', 'GF', 'GA']

    # Concatenate to get all games from each team's perspective
    all_team_games = pd.concat([home_games, away_games], ignore_index=True)

    # Iterate through each simulated season
    for season in all_team_games['SEASON'].unique():
        season_data = all_team_games[all_team_games['SEASON'] == season]
        print(f"Processing simulated season: {season}")

        # Iterate through each team in the current season
        for team in season_data['TEAM'].unique():
            team_games = season_data[season_data['TEAM'] == team]

            if len(team_games) < sample_size:
                print(f"  Skipping team {team} for season {season}: Not enough simulated games ({len(team_games)} games).")
                continue

            team_bootstrap_results = []
            for i in range(n_samples):
                sampled_games = team_games.sample(n=sample_size, replace=True, random_state=i)
                total_gf = sampled_games['GF'].sum()
                total_ga = sampled_games['GA'].sum()

                if (total_gf + total_ga) == 0:
                    gf_percent = 0.5 # Default to 0.5 if no goals (should be rare with 40 games)
                else:
                    gf_percent = total_gf / (total_gf + total_ga)

                team_bootstrap_results.append({
                    'season': season,
                    'team': team,
                    'gf_percent': gf_percent,
                    'sample_number': i + 1
                })
            all_bootstrap_samples.append(pd.DataFrame(team_bootstrap_results))
            print(f"  Completed bootstrapping for simulated team {team} in season {season} ({n_samples} samples).")

    if all_bootstrap_samples:
        final_df = pd.concat(all_bootstrap_samples, ignore_index=True)
        print("\nBootstrapping simulation on simulated data complete. Final DataFrame generated.")
        return final_df
    else:
        print("\nNo simulated data processed. Returning empty DataFrame.")
        return pd.DataFrame()
        

In [ ]:
if not full_season_schedule.empty:
    all_simulated_game_logs = []

    print(f"\nStarting main simulation for {num_simulated_seasons} seasons...")

    for i in range(num_simulated_seasons):
        print(f"\n--- Simulating Season {i+1} ---")
        # a. Draw 32 random values of GF% and assign to teams
        team_talent_distribution = assign_team_talent(
            gfdist_mean=gfdist_mean,
            gfdist_std=gfdist_std,
            goals_per_game_leagueavg=goals_per_game_leagueavg
        )

        # b. Simulate a full NHL regular season
        # We only need the game_log for this analysis, not the standings
        _, simulated_game_log = simulate_regular_season(
            full_season_schedule=full_season_schedule,
            team_averages=team_talent_distribution,
            home_ice_adv=home_ice_advantage,
            season_number=i+1
        )
        all_simulated_game_logs.append(simulated_game_log)

    # Concatenate all simulated game logs into a single DataFrame
    full_simulated_game_data = pd.concat(all_simulated_game_logs, ignore_index=True)
    print(f"\nTotal simulated games across all seasons: {len(full_simulated_game_data)}")

    # e. Repeat the bootstrapping draws from the simulated games
    bootstrapped_gf_percents_simulated = run_bootstrap_simulation_on_simulated_data(
        simulated_game_logs=full_simulated_game_data,
        n_samples=n_bootstrap_samples,
        sample_size=bootstrap_sample_size
    )

